# Lab 1 — Logistic regression from §1.3 to `nn.Sequential`

DATA 422 · *Hundred-Page Language Models Book* Ch. 1


We've already fit models in DATA 322 with `sklearn.linear_model` and `statsmodels`. Today we'll review our book's four supervised-learning steps applied to logistic regression, run that same example in the 322 style, then rebuild it in PyTorch using `nn.Sequential`.

## Learning objectives

By the end of this lab you can:

1. Name the four steps of supervised learning (§1.3) and write each one in math for logistic regression.
2. Fit a small two-feature example with `sklearn.linear_model.LogisticRegression` and `statsmodels.api.Logit` (DATA 322 tools).
3. Map $z = \mathbf{w}^\top \mathbf{x} + b$ and $\hat p = \sigma(z)$ onto `nn.Sequential(nn.Linear(...), nn.Sigmoid())`, then $\hat y = 1\{\hat p \ge 0.5\}$.
4. Train a Sequential model, read shapes, and plot BCE vs iteration.
5. Write the fitted prediction equation, score a new $\mathbf{x}$, and compare PyTorch with sklearn / statsmodels.


## New syntax from Ch. 1: PyTorch `Sequential`

| Math / idea | Code |
|-------------|------|
| tensor | `torch.tensor(...)`, `torch.from_numpy(...)` |
| $z = \mathbf{w}^\top \mathbf{x} + b$ | `nn.Linear(in_features, out_features)` |
| $\hat p = \sigma(z) = P(Y=1\mid\mathbf{x})$ (model) | `nn.Sigmoid()` (or `torch.sigmoid(z)`) |
| $\hat y \in \{0,1\}$ from cutoff $0.5$ | `(p_hat >= 0.5).int()` |
| stack layers | `nn.Sequential(nn.Linear(d, 1), nn.Sigmoid())` |
| BCE on **probabilities** $\hat p$ vs label $y$ | `nn.BCELoss()` — $y$ `float` in $\{0,1\}$, same shape as $\hat p$ |
| $\nabla_w \mathcal L$ | `loss.backward()` (fills `.grad`) |
| $w \leftarrow w - \eta \nabla$ | `optimizer.step()` |
| forget old grads | `optimizer.zero_grad()` |

DATA 322 reminders: `LinearRegression` / `LogisticRegression` from `sklearn.linear_model`; `sm.OLS` / `sm.Logit` from `statsmodels`.


## The four steps (§1.3), written for logistic regression

Burkov's supervised process has 4 simple steps. Let's write the process down for a specific example, a logistic regression model. Remember that logistic regression is the simplest example of a forward feed neural network. 

Notation we will stick to for the rest of the lab:

| Symbol | Meaning |
|--------|---------|
| $y \in \{0,1\}$ | observed **label** |
| $p = P(Y=1\mid \mathbf{x})$ | true probability of class 1 (unknown) |
| $\hat p = \sigma(z) \in (0,1)$ | **predicted probability** of class 1 |
| $\hat y \in \{0,1\}$ | **predicted label**, using cutoff $0.5$: $\hat y = 1$ if $\hat p \ge 0.5$, else $0$ |

**1. Get labeled data.** We have $n$ pairs $(\mathbf{x}_i, y_i)$ with $\mathbf{x}_i \in \mathbb{R}^{d}$ and binary labels $y_i \in \{0,1\}$.

**2. Choose a model** (a family of functions with parameters). Logistic regression uses a linear combination of the inputs, then the sigmoid to get a probability:

$$
z = \mathbf{w}^\top \mathbf{x} + b, \qquad
\hat p = \sigma(z) = \frac{1}{1+e^{-z}} \in (0,1).
$$

Here $\mathbf{w} \in \mathbb{R}^{d}$ and $b \in \mathbb{R}$ are the **parameters**. $z$ is unconstrained; $\hat p$ estimates $p$. Then

$$
\hat y = \begin{cases} 1 & \text{if }\hat p \ge 0.5, \\ 0 & \text{otherwise.} \end{cases}
$$

**3. Choose a loss** that is small when $\hat p$ is close to the **label** $y$ (not to $\hat y$). Binary cross-entropy for one example is equivalent to using the method of Maximum Likelihood with a Binomial distribution for the labels:

$$
\mathcal L_i = -\bigl[ y_i \log \hat p_i + (1-y_i)\log(1-\hat p_i) \bigr],
$$

and the sample mean $\mathcal L = \frac{1}{n}\sum_i \mathcal L_i$.

**4. Find parameters that make the loss small.** In DATA 322 this step was inside `.fit()`. PyTorch makes the update explicit: gradient descent

$$
\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla_{\mathbf{w}}\mathcal L, \qquad
b \leftarrow b - \eta \nabla_{b}\mathcal L.
$$



## Demo 0 — environment and the shared example


In [1]:
%pip install -q plotnine

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.linear_model import LinearRegression, LogisticRegression
import statsmodels.api as sm
from plotnine import (
    ggplot, aes, geom_point, geom_line, geom_raster,
    scale_color_manual, scale_shape_manual, scale_fill_gradient2,
    labs, theme_bw, coord_equal,
)
from IPython.display import Markdown, display

torch.manual_seed(422)
np.random.seed(422)
print("torch", torch.__version__)
print("numpy", np.__version__)
print("device", "cuda" if torch.cuda.is_available() else "cpu")
# Desired: a version string, and usually `cpu` on Colab unless you picked a GPU.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'torch'

**Step 1 — the data.** We'll start with two Normal/Gaussian features, living in $\mathbb{R}^{2}$. There will be two classes with labels $y=0$ or $y=1$. Axes are the two features $x_1$ and $x_2$; color and shape are the observed label $y$. We keep a NumPy copy for sklearn/statsmodels and later a tensor copy for PyTorch.


In [ ]:
rng = np.random.default_rng(422)
n_per = 200
cov = np.array([[1.0, 0.3], [0.3, 1.0]])
X0 = rng.multivariate_normal([-2.0, -2.0], cov, n_per)
X1 = rng.multivariate_normal([2.0, 2.0], cov, n_per)
X_np = np.vstack([X0, X1]).astype(np.float64)
y_np = np.hstack([np.zeros(n_per), np.ones(n_per)]).astype(np.float64)
perm = rng.permutation(X_np.shape[0])
X_np, y_np = X_np[perm], y_np[perm]

print("X_np: shape", X_np.shape, "dtype", X_np.dtype)
print("y_np: shape", y_np.shape, "dtype", y_np.dtype, "mean (prevalence)", round(float(y_np.mean()), 3))
print("first 8 feature rows X_np:\n", np.round(X_np[:8], 3))
print("first 8 labels y:", y_np[:8].astype(int))
# Desired: X (400, 2), y mean 0.5

pts = pd.DataFrame({
    "x1": X_np[:, 0],
    "x2": X_np[:, 1],
    "y": pd.Categorical(y_np.astype(int).astype(str)),
})
print("pts (plot data) shape", pts.shape, "columns", list(pts.columns))
print(pts.head())

(
    ggplot(pts, aes(x="x1", y="x2", color="y", shape="y"))
    + geom_point(size=2.4, alpha=0.85)
    + scale_color_manual(values={"0": "#3b4cc0", "1": "#b40426"}, name="label $y$")
    + scale_shape_manual(values={"0": "o", "1": "^"}, name="label $y$")
    + labs(
        x="$x_1$",
        y="$x_2$",
        title="Observed data: two features, color/shape = label $y$",
    )
    + coord_equal()
    + theme_bw()
)


## DATA 322 style — sklearn and statsmodels (steps 2–4, hidden in `.fit`)

In 322 you used **`LinearRegression`** for a continuous target:

$$
\text{numeric fit} = \mathbf{w}^\top \mathbf{x} + b
$$

(`sklearn.linear_model.LinearRegression`, or `sm.OLS`). Logistic regression uses the same linear combination of features and weights with a bias but uses the sigmoid to tranform to $\hat p$.  We'll use a $0.5$ cutoff for converting the predicted probability to a predicted label $\hat y$. `LogisticRegression` is in the sklearn module. `sm.Logit` is the statsmodels analogue of `sm.OLS`.



In [ ]:
# DATA 322 regression API (same module). OLS treats y as a number, not p or ŷ.
lin = LinearRegression().fit(X_np, y_np)
print("LinearRegression object:", lin)
print("  intercept_ shape", np.shape(lin.intercept_), "value", round(float(lin.intercept_), 3))
print("  coef_      shape", lin.coef_.shape, "value", np.round(lin.coef_, 3))

# Logistic: p̂ = σ(Xw + b), ŷ = 1{p̂ ≥ 0.5}. C=np.inf ≈ unregularized MLE.
clf = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=2000)
clf.fit(X_np, y_np.astype(int))
print("\nLogisticRegression object:", clf)
print("  coef_      shape", clf.coef_.shape, "w =", np.round(clf.coef_[0], 3))
print("  intercept_ shape", np.shape(clf.intercept_), "b =", round(float(clf.intercept_[0]), 3))

p_hat_sk = clf.predict_proba(X_np)[:, 1]   # predicted P(Y=1 | x)
y_hat_sk = (p_hat_sk >= 0.5).astype(int)
print("\npredict_proba[:, 1]  shape", p_hat_sk.shape)
print("first 8  y     ", y_np[:8].astype(int))
print("first 8  p_hat ", np.round(p_hat_sk[:8], 3))
print("first 8  y_hat ", y_hat_sk[:8], "  (cutoff 0.5)")
sk_acc = float((y_hat_sk == y_np.astype(int)).mean())
print("sklearn accuracy P(ŷ = y) =", round(sk_acc, 3))
# Desired: acc around 0.99 (blobs are well separated).


In [ ]:
# DATA 322 used sm.OLS(y, sm.add_constant(X)).fit()
# Binary y → Logit. add_constant supplies the intercept b.
X_sm = sm.add_constant(X_np)
print("X_sm shape (intercept column first)", X_sm.shape)
print("X_sm[:4]:\n", np.round(X_sm[:4], 3))

logit = sm.Logit(y_np, X_sm).fit(disp=False)
print("\nstatsmodels params (b, w1, w2):\n", logit.params.round(3))
p_hat_sm = logit.predict(X_sm)
y_hat_sm = (p_hat_sm >= 0.5).astype(int)
print("p_hat_sm shape", p_hat_sm.shape)
print("first 8  y     ", y_np[:8].astype(int))
print("first 8  p_hat ", np.round(p_hat_sm[:8], 3))
print("first 8  y_hat ", y_hat_sm[:8])
sm_acc = float((y_hat_sm == y_np.astype(int)).mean())
print("statsmodels accuracy", round(sm_acc, 3), "| sklearn accuracy", round(sk_acc, 3))
# Desired: acc again ~0.99, and close to sklearn.
# Coefs need not match sklearn to three decimals (different solvers).


## Exercise 1 — one example by hand (step 2)

For $\mathbf{w}=[2,-1]$, $b=0$, $\mathbf{x}=[0.5, 1.0]$:

1. Compute $z = \mathbf{w}^\top \mathbf{x} + b$ and $\hat p = \sigma(z)$ on paper.
2. Then $\hat y = 1$ if $\hat p \ge 0.5$, else $0$.
3. Fill the tensors below.

**Desired (rounded):** $z = 0.0$, $\hat p = 0.5$, and $\hat y = 1$ (the cutoff includes $0.5$).


In [ ]:
w = torch.tensor([2.0, -1.0])
b = torch.tensor(0.0)
x = torch.tensor([0.5, 1.0])
print("w", w, "shape", tuple(w.shape))
print("b", b, "shape", tuple(b.shape))
print("x", x, "shape", tuple(x.shape))

z = None          # TODO: w.dot(x) + b
p_hat = None      # TODO: torch.sigmoid(z)
y_hat = None      # TODO: (p_hat >= 0.5).int()   # predicted label
print("z", z)
print("p_hat", p_hat)
print("y_hat", y_hat)


## PyTorch from the basics — only `nn.Sequential`

We now write steps 2–4 ourselves. The book stacks a linear map and a sigmoid:

```
x  →  Linear (z = Wx + b)  →  Sigmoid (p̂ = σ(z))  →  p̂
```

Then $\hat y = 1\{\hat p \ge 0.5\}$. 


### Demo 1 — tensors and shapes

A tensor is a typed multidimensional array. `nn.Linear(d, 1)` maps a batch of $d$-dimensional rows to a batch of scalar $z$ values. The batch size is an extra leading dimension. Printing shapes is a good way to check your understanding of how the code implements the math.


In [ ]:
layer = nn.Linear(4, 1)
x = torch.randn(8, 4)          # (batch, features)
z = layer(x)                   # (8, 1) 
print("layer:\n", layer)
print("weight shape", tuple(layer.weight.shape), "\n", layer.weight.data)
print("bias   shape", tuple(layer.bias.shape), layer.bias.data)
print("x shape", tuple(x.shape), "\nfirst row", x[0])
print("z shape", tuple(z.shape), "\nfirst 4 z", z[:4].detach().squeeze())
print("n_params", sum(p.numel() for p in layer.parameters()))
# Desired: z (8, 1)   n_params 5   because 4 weights + 1 bias


### Demo 2 — `Sequential` is the logistic model

`nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())` **is** $\hat p = \sigma(\mathbf{w}^\top \mathbf{x} + b)$. The `2` is $d$; the `1` is one $\hat p$ per row. After the sigmoid, values are in $(0,1)$: this tensor is $\hat p$.


In [ ]:
model = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())
x = torch.randn(4, 2)
p_hat = model(x)
print("model:\n", model)
print("x shape", tuple(x.shape))
print("p_hat shape", tuple(p_hat.shape), "min/max", round(float(p_hat.min()), 3), round(float(p_hat.max()), 3))
print("p_hat (predicted probabilities):\n", p_hat.detach())
y_hat = (p_hat >= 0.5).int()
print("y_hat (predicted labels, cutoff 0.5):\n", y_hat)
# Desired: p_hat shape (4, 1); values between 0 and 1.
# Printed modules: Linear then Sigmoid.


### Demo 3 — the training loop (steps 3–4)

Loss: `nn.BCELoss()` on **probabilities** $\hat p$ vs labels $y$. Loop:

1. `optimizer.zero_grad()` — forget last step's $\nabla$.
2. $\hat p =$ `model(x)` — forward.
3. `loss = criterion(p_hat, y)` then `loss.backward()` — fills `.grad`.
4. `optimizer.step()` — $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla$.

$y$ must be `float` and the **same shape** as $\hat p$ (here `(n, 1)`). $\hat y$ is not used in the loss.


In [ ]:
torch.manual_seed(422)
model = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())
x = torch.tensor([[0.0, 0.0], [1.0, 1.0]])
y = torch.tensor([[0.0], [1.0]])          # (2, 1) to match Sequential
print("x shape", tuple(x.shape), "\n", x)
print("y shape", tuple(y.shape), "\n", y)
criterion = nn.BCELoss()
opt = torch.optim.SGD(model.parameters(), lr=0.5)

for step in range(40):
    opt.zero_grad()
    p_hat = model(x)
    loss = criterion(p_hat, y)
    loss.backward()
    opt.step()

y_hat = (p_hat.detach() >= 0.5).int()
print("final BCE", round(loss.item(), 4))
print("p_hat", p_hat.detach().squeeze().round(decimals=3).tolist())
print("y_hat", y_hat.squeeze().tolist(), "  vs y", y.squeeze().int().tolist())
# Desired: loss clearly below ~0.7. p_hat moves toward [0, 1]; y_hat matches y.


### Demo 4 — the same blob example as sklearn / statsmodels

Convert the NumPy arrays from Demo 0. Train Sequential + `BCELoss`, recording BCE every iteration. Then we will plot the loss, write the fitted equation, score a new point, and redraw the scatter with $\hat p$ in the background.


In [ ]:
torch.manual_seed(422)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)   # (400, 1)
print("X (tensor) shape", tuple(X.shape), "dtype", X.dtype)
print("y (tensor) shape", tuple(y.shape), "dtype", y.dtype, "  ← labels, not probabilities")
print("X[:5]:\n", X[:5])
print("y[:5].T", y[:5].squeeze().tolist())

model = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())
print("\nmodel before training:\n", model)
for name, p in model.named_parameters():
    print(name, "shape", tuple(p.shape), "\n", p.detach())

criterion = nn.BCELoss()
opt = torch.optim.SGD(model.parameters(), lr=0.5)
print("criterion", criterion)
print("optimizer", opt)

losses = []
for step in range(400):
    opt.zero_grad()
    p_hat = model(X)
    loss = criterion(p_hat, y)
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if step % 100 == 0 or step == 399:
        print(f"step {step:3d}  BCE={loss.item():.4f}  p_hat[:5]={p_hat[:5].detach().squeeze().round(decimals=3).tolist()}")

print("\np_hat shape", tuple(p_hat.shape), "  ← predicted P(Y=1)")
y_hat = (p_hat.detach() >= 0.5).float()
print("y_hat shape", tuple(y_hat.shape), "  ← predicted labels, cutoff 0.5")
pt_acc = float((y_hat == y).float().mean())
print("pytorch accuracy P(ŷ = y) =", round(pt_acc, 3),
      "| sklearn", round(sk_acc, 3), "| statsmodels", round(sm_acc, 3))
# Desired: pytorch acc around 0.99, within a point or so of sklearn and statsmodels.


Plot the recorded BCE against iteration. You should see the loss drop quickly, then flatten. That is step 4 (optimization) becoming visible.

In [ ]:
loss_df = pd.DataFrame({"iteration": np.arange(len(losses)), "BCE": losses})
print("loss_df shape", loss_df.shape)
print("first 3 rows:\n", loss_df.head(3))
print("last 3 rows:\n", loss_df.tail(3))

(
    ggplot(loss_df, aes(x="iteration", y="BCE"))
    + geom_line(color="#1f4e79", size=0.8)
    + labs(
        x="iteration",
        y="BCE",
        title="Training loop: BCE vs iteration",
    )
    + theme_bw()
)

Print the fitted $\mathbf{w}$ and $b$, then write the prediction rule in math. For two features,

$$
z = w_1 x_1 + w_2 x_2 + b, \qquad \hat p = \sigma(z), \qquad \hat y = 1\{\hat p \ge 0.5\}.
$$

In [ ]:
linear = model[0]
w = linear.weight.detach().squeeze()
b = float(linear.bias.detach())
w1, w2 = float(w[0]), float(w[1])
print("model[0] (Linear):", linear)
print("weight tensor shape", tuple(linear.weight.shape), "→ w =", [round(w1, 4), round(w2, 4)])
print("bias   tensor shape", tuple(linear.bias.shape), "→ b =", round(b, 4))
print("model[1] (Sigmoid) has no parameters:", model[1])

display(Markdown(rf"""
The fitted predictor (plug in any $x_1,x_2$) is

$$
z = {w1:.3f}\, x_1 + {w2:.3f}\, x_2 + ({b:.3f}),
\qquad
\hat p = \sigma(z) = \frac{{1}}{{1+\exp(-z)}}.
$$

The predicted label uses cutoff $0.5$:

$$
\hat y =
\begin{{cases}}
1 & \text{{if }}\hat p \ge 0.5,\\
0 & \text{{otherwise.}}
\end{{cases}}
$$

Weights will not match sklearn to three decimals (SGD vs L-BFGS). The **boundary** $\hat p=0.5$ (i.e. $z=0$) should look similar.
"""))

Pick two new feature values $(x_1, x_2)$, run them through the **same** fitted model, and read off $\hat p$ then $\hat y$. The input shape is `(1, 2)`: one row, two features. We also compute $z$ from the linear layer alone so you can see the two pieces.

In [ ]:
# One new observation: two feature values. Shape must be (1, 2) — a batch of one.
x_new = np.array([[0.0, 0.5]], dtype=np.float32)
x_new_t = torch.tensor(x_new)
print("x_new numpy shape", x_new.shape, "values", x_new)
print("x_new tensor shape", tuple(x_new_t.shape), "\n", x_new_t)

linear = model[0]
z_new = linear(x_new_t)
p_hat_new = model(x_new_t)
y_hat_new = (p_hat_new >= 0.5).int()
print("z    shape", tuple(z_new.shape), "value", round(float(z_new.detach()), 4), "  ← logit")
print("p_hat shape", tuple(p_hat_new.shape), "value", round(float(p_hat_new.detach()), 4), "  ← predicted P(Y=1)")
print("y_hat shape", tuple(y_hat_new.shape), "value", int(y_hat_new.item()), "  ← predicted label, cutoff 0.5")

# Same number from the written equation (should match p_hat).
z_by_hand = w1 * float(x_new[0, 0]) + w2 * float(x_new[0, 1]) + b
p_by_hand = 1.0 / (1.0 + np.exp(-z_by_hand))
print("by-hand z", round(z_by_hand, 4), "by-hand p_hat", round(p_by_hand, 4))

Same scatter as Step 1, but the **background** is $\hat p$ from the fitted Sequential model (blue $\approx 0$, red $\approx 1$). The black line is the decision boundary $z=0$ (equivalently $\hat p = 0.5$). Points still show the observed label $y$. The × is the new $\mathbf{x}$ from the previous cell.

In [ ]:
# Grid over the same (x1, x2) window as the scatter.
pad = 0.6
n_grid = 80
x1_vals = np.linspace(X_np[:, 0].min() - pad, X_np[:, 0].max() + pad, n_grid)
x2_vals = np.linspace(X_np[:, 1].min() - pad, X_np[:, 1].max() + pad, n_grid)
xx1, xx2 = np.meshgrid(x1_vals, x2_vals)
grid = np.column_stack([xx1.ravel(), xx2.ravel()]).astype(np.float32)
print("grid X shape", grid.shape, "  (n_grid × n_grid rows of (x1, x2))")

with torch.no_grad():
    p_hat_grid = model(torch.tensor(grid)).numpy().ravel()
print("p_hat_grid shape", p_hat_grid.shape, "min/max", round(float(p_hat_grid.min()), 3), round(float(p_hat_grid.max()), 3))

grid_df = pd.DataFrame({"x1": xx1.ravel(), "x2": xx2.ravel(), "p_hat": p_hat_grid})
new_df = pd.DataFrame({"x1": [x_new[0, 0]], "x2": [x_new[0, 1]]})
# Boundary z = 0  ⇔  p̂ = 0.5  ⇔  w1 x1 + w2 x2 + b = 0
x1_b = np.linspace(x1_vals.min(), x1_vals.max(), 200)
x2_b = -(w1 / w2) * x1_b - b / w2
bd = pd.DataFrame({"x1": x1_b, "x2": x2_b})
bd = bd[(bd["x2"] >= x2_vals.min()) & (bd["x2"] <= x2_vals.max())]
print("boundary line rows", len(bd))

(
    ggplot()
    + geom_raster(grid_df, aes(x="x1", y="x2", fill="p_hat"), interpolate=True)
    + geom_line(bd, aes(x="x1", y="x2"), color="black", size=0.7)
    + geom_point(pts, aes(x="x1", y="x2", color="y", shape="y"), size=2.4, alpha=0.95, stroke=0.4)
    + geom_point(new_df, aes(x="x1", y="x2"), color="black", size=4, shape="x", stroke=1.2)
    + scale_fill_gradient2(
        low="#3b4cc0", mid="white", high="#b40426", midpoint=0.5,
        limits=(0, 1), name=r"$\hat p$",
    )
    + scale_color_manual(values={"0": "#1a1a2e", "1": "#6b0000"}, name="label $y$")
    + scale_shape_manual(values={"0": "o", "1": "^"}, name="label $y$")
    + labs(
        x="$x_1$",
        y="$x_2$",
        title=r"Background = $\hat p$; points = observed $y$; black line = $\hat p=0.5$; × = new x",
    )
    + coord_equal()
    + theme_bw()
)

## Exercise 2 — complete the Sequential loop

Fill every `None`. 


In [ ]:
torch.manual_seed(422)
model = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())
x = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y = torch.tensor([[0.0], [0.0], [0.0], [1.0]])   # AND labels
print("x shape", tuple(x.shape), "y shape", tuple(y.shape))
criterion = nn.BCELoss()
opt = torch.optim.SGD(model.parameters(), lr=1.0)

loss = None
for step in range(80):
    opt.zero_grad()
    p_hat = None     # TODO: model(x)
    loss = None      # TODO: criterion(p_hat, y)
    # TODO: backward and step

print("loss", None if loss is None else round(loss.item(), 4))
# Self-check: loss < 0.4 after 80 steps with seed 422.


## Exercise 3 — parameter count

`nn.Sequential(nn.Linear(6, 1), nn.Sigmoid())`: how many **trainable** numbers? (`Sigmoid` has none.)

**Desired:** `7`.


In [ ]:
m = nn.Sequential(nn.Linear(6, 1), nn.Sigmoid())
n = sum(p.numel() for p in m.parameters())
print(n)



## Exercise 4 — tiny-batch overfit

Train Sequential + `BCELoss` on **these 8 rows** until accuracy at cutoff $0.5$ is 1.0. Accuracy is $P(\hat y = y)$, where $\hat y = 1\{\hat p \ge 0.5\}$. If you cannot, the wiring is wrong (shape of $y$, forgot `backward`/`step`, …).

**Desired:** `acc = 1.0`.


In [ ]:
torch.manual_seed(422)
x = torch.randn(8, 3)
y = (x[:, 0] > 0).float().unsqueeze(1)
print("x shape", tuple(x.shape), "y shape", tuple(y.shape), "y", y.squeeze().int().tolist())
model = nn.Sequential(nn.Linear(3, 1), nn.Sigmoid())
opt = torch.optim.SGD(model.parameters(), lr=1.0)
crit = nn.BCELoss()

for _ in range(300):
    opt.zero_grad()
    p_hat = model(x)
    loss = crit(p_hat, y)
    loss.backward()
    opt.step()

y_hat = (p_hat.detach() >= 0.5).float()
acc = float((y_hat == y).float().mean())
print("p_hat", p_hat.detach().squeeze().round(decimals=3).tolist())
print("y_hat", y_hat.squeeze().int().tolist())
print("loss", round(loss.item(), 4), "acc", acc)
# Desired: acc == 1.0
